In [ ]:
%load_ext kedro.ipython

In [ ]:
!uv pip install --upgrade seaborn matplotlib

In [ ]:
df = catalog.load("ortodoncja")

df

In [ ]:
df["growth direction"].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def plot_distributions(dataframe):
    numeric_cols = dataframe.select_dtypes(include=['number']).columns

    for col in numeric_cols:
        fig, (ax_box, ax_hist) = plt.subplots(
            2, 
            sharex=True, 
            gridspec_kw={"height_ratios": (.15, .85)}, 
            figsize=(10, 6)
        )
        
        sns.boxplot(x=dataframe[col], ax=ax_box, color="skyblue")
        ax_box.set(xlabel='')
        
        sns.histplot(data=dataframe, x=col, ax=ax_hist, kde=True, bins=30, color="skyblue")
        ax_hist.set(ylabel='Liczba wystąpień (Count)')
        
        plt.suptitle(f'Rozkład zmiennej: {col}', fontsize=16, fontweight='bold', y=0.95)
        
        plt.tight_layout()
        plt.show()

plot_distributions(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()

enc.fit(df[["growth direction"]])

df["growth direction"] = enc.transform(df[["growth direction"]])

df


In [ ]:
enc.classes_

In [ ]:
def increment_feature_engineering(df: pd.DataFrame):
    all_columns: list[str] = list(df.columns)

    nine_y_columns = [column for column in all_columns if column.startswith("9_")]
    twelve_y_columns = [column for column in all_columns if column.startswith("12_")]

    parameter_names = [column[2:] for column in nine_y_columns]

    for parameter_name in parameter_names:
        df[parameter_name + "_diff"] = df["12_" + parameter_name] - df["9_" + parameter_name]
        df[parameter_name + "_rat"] = df["12_" + parameter_name] / df["9_" + parameter_name]

    df.drop(columns=twelve_y_columns, inplace=True)
    
increment_feature_engineering(df)

In [ ]:
corr = df.corr(numeric_only=True)

corr

In [ ]:
def correlated_columns_cleanup(df: pd.DataFrame, corr_matrix: pd.DataFrame, target_column_name: str, threshold=0.8):
    columns = list(corr_matrix.columns)
    columns.remove(target_column_name)
    columns_len = len(columns)

    columns_to_remove = []

    for first_column_idx in range(0, columns_len):
        for second_column_idx in range(first_column_idx + 1, columns_len):
            first_column_name = columns[first_column_idx]
            second_column_name = columns[second_column_idx]
            
            if abs(corr_matrix.loc[first_column_name, second_column_name]) < threshold:
                continue

            column_name_to_remove = first_column_name if abs(corr_matrix.loc[first_column_name, target_column_name]) > abs(corr_matrix.loc[second_column_name, target_column_name]) else second_column_name

            columns_to_remove.append(column_name_to_remove)

    print(columns_to_remove)

    df.drop(columns=columns_to_remove, inplace=True)


# correlated_columns_cleanup(df, corr, "growth direction")

In [ ]:
def find_outliers_iqr(df, kolumny_do_sprawdzenia, mnoznik=1.5):
    raport_outlierow = {}
    
    for kolumna in kolumny_do_sprawdzenia:
        Q1 = df[kolumna].quantile(0.25)
        Q3 = df[kolumna].quantile(0.75)
        IQR = Q3 - Q1
        
        dolna_granica = Q1 - (mnoznik * IQR)
        gorna_granica = Q3 + (mnoznik * IQR)
        
        outliery = df[(df[kolumna] < dolna_granica) | (df[kolumna] > gorna_granica)]
        
        if not outliery.empty:
            print(f"--- Kolumna: {kolumna} ---")
            print(f"Znaleziono: {len(outliery)} potencjalnych outlierów.")
            print(f"Granice normy: od {dolna_granica:.2f} do {gorna_granica:.2f}")
            
            print(outliery[kolumna].sort_values())
            print("\n")
            
            raport_outlierow[kolumna] = outliery
        else:
            print(f"--- Kolumna: {kolumna} ---")
            print("Czysto! Brak outlierów.\n")
            
    return raport_outlierow

columns_to_check = list(df.columns)

columns_to_check.remove('growth direction')

find_outliers_iqr(df, columns_to_check)

In [ ]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(contamination='auto', random_state=42)

outlier_labels = iso_forest.fit_predict(df.drop(columns=['growth direction']))

df_czyste = df[outlier_labels == 1]

print(f"Isolation Forest wyrzucił {len(df) - len(df_czyste)} pacjentów.")

In [ ]:
df = df_czyste

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

print("Seaborn:", sns.__version__)
print("Matplotlib:", matplotlib.__version__)

corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(20, 20))

sns.heatmap(
    corr_matrix, 
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title("Macierz Korelacji", fontsize=16)
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))

res = cross_validate(model,df.drop(columns="growth direction"), df["growth direction"])

res['test_score']

In [ ]:
import numpy as np

print("Srednia: ", np.mean(res['test_score']))
print("Odch: ", np.std(res['test_score']))

In [ ]:
# Bez corr cleanup z usunietymi 12: 0.709 +- 0.0353
# Z corr cleanup z usunietymi 12: 0.695 +- 0.0381
# Z corr cleanup z zostawionymi 12: 0.690 +- 0.0358